# Visual Data Science Notebook on Renewable Energy Insight
This notebook helps to work through the pipeline of getting interesting information and metrics on renewable energies

## Return on investment of different countries

In [8]:
import pandas as pd
import numpy as np
import chardet
DATA_FOLDER = "data/"
SELECTED_COUNTRIES = ["Austria"]

In [13]:
# with open(DATA_FOLDER + 'capacity-generation-emission.csv', 'rb') as f:
#     result = chardet.detect(f.read())
#     encoding = result['encoding']
capacity_generation_emission_content = pd.read_csv(DATA_FOLDER + "capacity-generation-emission.csv", encoding='utf-8')
#print(capacity_generation_emission_content.head())

# with open(DATA_FOLDER + 'public-investments.csv', 'rb') as f:
#     result = chardet.detect(f.read())
#     encoding = result['encoding']
# print(f"Detected encoding: {encoding}")
public_investment_content = pd.read_csv(DATA_FOLDER + 'public-investments.csv', encoding='iso-8859-1')
print(public_investment_content.head())
print(public_investment_content.columns)

# with open(DATA_FOLDER + 'european-energy-prices.csv', 'rb') as f:
#     result = chardet.detect(f.read())
#     encoding = result['encoding']
european_energy_prices_content = pd.read_csv(DATA_FOLDER + 'european-energy-prices.csv', encoding='iso-8859-1')
#print(european_energy_prices_content.head())







  Country/area                  Technology  Year  \
0  Afghanistan  On-grid solar photovoltaic  2000   
1  Afghanistan  On-grid solar photovoltaic  2001   
2  Afghanistan  On-grid solar photovoltaic  2002   
3  Afghanistan  On-grid solar photovoltaic  2003   
4  Afghanistan  On-grid solar photovoltaic  2004   

  Public Investments (2022 Million USD)  
0                                     -  
1                                  0.10  
2                                     -  
3                                     -  
4                                     -  
Index(['Country/area', 'Technology', 'Year',
       'Public Investments (2022 Million USD)'],
      dtype='object')


In [4]:
def get_average_european_energy_price(country_list):
    country_list_df = pd.DataFrame()
    for country in country_list:
        country_data = european_energy_prices_content[european_energy_prices_content['Country'] == country].copy()
        country_data['Year'] = pd.to_datetime(country_data['Date']).dt.year
        country_data['Avarage Price'] = country_data.groupby(['Year'])['Price (EUR/MWhe)'].transform('mean')
        country_data = country_data.drop(['Date', 'Price (EUR/MWhe)'], axis=1)
        country_data = country_data.drop_duplicates()
        country_list_df = pd.concat([country_list_df, country_data], ignore_index=True)
       

    return country_list_df

In [78]:
country_prices = [get_average_european_energy_price(country) for country in [["Austria","Germany"]]]
print(country_prices)

[    Country ISO3 Code  Year  Avarage Price
0   Austria       AUT  2015      31.764167
1   Austria       AUT  2016      28.956667
2   Austria       AUT  2017      34.419167
3   Austria       AUT  2018      40.891667
4   Austria       AUT  2019      40.137500
5   Austria       AUT  2020      33.090833
6   Austria       AUT  2021     107.598333
7   Austria       AUT  2022     262.435000
8   Austria       AUT  2023     102.389167
9   Austria       AUT  2024      81.182500
10  Austria       AUT  2025      95.840000
11  Germany       DEU  2015      31.764167
12  Germany       DEU  2016      28.956667
13  Germany       DEU  2017      34.638333
14  Germany       DEU  2018      43.463333
15  Germany       DEU  2019      37.814167
16  Germany       DEU  2020      30.372500
17  Germany       DEU  2021      97.275000
18  Germany       DEU  2022     235.530833
19  Germany       DEU  2023      95.390833
20  Germany       DEU  2024      77.746667
21  Germany       DEU  2025      88.682000]


In [36]:
def get_yearly_public_investment(country_list):
    country_list_df = pd.DataFrame()
    for country in country_list:
        country_data = public_investment_content[public_investment_content['Country/area'] == country].copy()
        country_list_df = pd.concat([country_list_df, country_data], ignore_index=True)
    return country_list_df

def get_countries_with_most_data(year=None, top_n=10, technology='Multiple renewables'):
    # Count countries that have a non-missing / non-'-' value in the Public Investments column.
    col = 'Public Investments (2022 Million USD)'
    df = public_investment_content.copy()

    # filter by technology
    df = df[df['Technology'] == technology]

    # optional year filter if the dataset has a Year column
    if year is not None and 'Year' in df.columns:
        df = df[df['Year'] == year]

    # consider non-null and not the hyphen placeholder
    valid = df[df[col].notna() & (df[col] != '-')]

    # count valid rows per country and return top N countries
    top_countries = valid['Country/area'].value_counts().head(top_n).index.tolist()
    num_valid_rows = valid['Country/area'].value_counts().head(top_n).tolist()
    top_countries = (top_countries, num_valid_rows)
    return top_countries

In [ ]:
investments = get_yearly_public_investment(SELECTED_COUNTRIES)
print(investments)

top_countries = get_countries_with_most_data(top_n=50)
print(top_countries)

    Country/area                  Technology  Year  \
0        Austria  On-grid solar photovoltaic  2000   
1        Austria  On-grid solar photovoltaic  2001   
2        Austria  On-grid solar photovoltaic  2002   
3        Austria  On-grid solar photovoltaic  2003   
4        Austria  On-grid solar photovoltaic  2004   
..           ...                         ...   ...   
403      Austria         Multiple renewables  2019   
404      Austria         Multiple renewables  2020   
405      Austria         Multiple renewables  2021   
406      Austria         Multiple renewables  2022   
407      Austria         Multiple renewables  2023   

    Public Investments (2022 Million USD)  
0                                       -  
1                                       -  
2                                       -  
3                                       -  
4                                       -  
..                                    ...  
403                                     -  